# Description
Extract and summarize disruption narratives for Layer 3 analysis.

# Input
data/metadata/analysis_corpus_labeled.csv

# Output
results/layer3_index_by_month_labeled.csv and layer3 figures in figures/



# 05 Layer 3: Disruption Narrative Analysis

## Description
Extracts sentences containing Layer 3 disruption/warfare terms (disinformation, propaganda,
false flag, information operations, hybrid warfare, psyops) and analyzes their co-occurrence
with value vocabulary terms.

Three outputs are produced:
1. **Sentence extraction** – All sentences matching Layer 3 lexicon, with co-occurring value terms
2. **Intensity time series** – Monthly count of disruption sentences by actor × country
3. **Co-occurrence heatmaps** – Layer 3 term × value term co-occurrence by country

## Input
- `corpus/analysis_corpus_labeled.csv` – Labeled analysis corpus

## Output
- `outputs/layer3_sentences_labeled.csv` – Extracted disruption sentences with metadata
- `outputs/layer3_index_by_month_labeled.csv` – Monthly disruption intensity index
- `outputs/fig_layer3_intensity_timeseries_labeled.png` – Time series of disruption intensity
- `outputs/fig_layer3_value_cooccur_{country}.png` – Co-occurrence heatmaps (Russia, Ukraine, Georgia, Institution)


In [ ]:
import os, re
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

INPUT   = "./corpus/analysis_corpus_labeled.csv"
OUT_DIR = "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
# Layer 3 disruption lexicon
L3_TERMS = {
    "disinformation": [r"\bdisinformation\b", r"\bmisinformation\b"],
    "propaganda":     [r"\bpropaganda\b"],
    "false_flag":     [r"\bfalse flag\b", r"\bfalse-flag\b"],
    "provocation":    [r"\bprovocation\b", r"\bprovocative\b"],
    "info_ops":       [r"\binformation war(?:fare)?\b", r"\binformation operations?\b"],
    "hybrid_war":     [r"\bhybrid war(?:fare)?\b"],
    "psyops":         [r"\bpsyops\b", r"\bpsychological operations?\b"],
}

# Value lexicon (for co-occurrence analysis)
VALUES = {
    "sovereignty":           [r"\bsovereignty\b"],
    "territorial_integrity": [r"\bterritorial integrity\b"],
    "democracy":             [r"\bdemocracy\b", r"\bdemocratic\b"],
    "security":              [r"\bsecurity\b"],
    "rules_based":           [r"\brules[- ]based\b"],
}

def compile_map(m): return {k: re.compile("|".join(v), flags=re.I) for k,v in m.items()}
RX_L3  = compile_map(L3_TERMS)
RX_VAL = compile_map(VALUES)

def load_text(p):
    try:
        with open(p, "r", encoding="utf-8") as f: return f.read()
    except Exception: return ""

def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    sents = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sents if len(s.strip()) >= 20]

def hits(rx_map, s): return [k for k,rx in rx_map.items() if rx.search(s)]


In [ ]:
# Extract Layer 3 sentences
df = pd.read_csv(INPUT, dtype=str)
df = df[df["clean_path"].notna()].copy()
rows = []
for _, r in df.iterrows():
    text = load_text(r["clean_path"])
    if not text: continue
    month = str(r.get("month", r.get("date","")))[:7]
    for sent in split_sentences(text):
        l3h = hits(RX_L3, sent)
        if not l3h: continue
        valh = hits(RX_VAL, sent)
        rows.append({"doc_id": r.get("doc_id",""), "url": r.get("url",""),
                     "date": r.get("date",""), "month": month,
                     "actor": r.get("actor",""), "country": r.get("country",""),
                     "sentence": sent, "l3_terms": ",".join(l3h), "value_terms": ",".join(valh)})
out = pd.DataFrame(rows)
out.to_csv(f"{OUT_DIR}/layer3_sentences_labeled.csv", index=False)
print(f"Saved layer3_sentences_labeled.csv  rows={len(out)}")


In [ ]:
# Monthly intensity index and time series plot
l3 = pd.read_csv(f"{OUT_DIR}/layer3_sentences_labeled.csv", dtype=str)
g = l3.groupby(["actor","country","month"]).agg(
    n_l3_sents=("sentence","count"),
    n_docs=("doc_id", pd.Series.nunique),
).reset_index()
g.to_csv(f"{OUT_DIR}/layer3_index_by_month_labeled.csv", index=False)
print(f"Saved layer3_index_by_month_labeled.csv")

TARGETS = [("State","Russia","firebrick","-"),("State","Ukraine","royalblue","-"),
           ("State","Georgia","seagreen","-"),("Institution","Institution","darkorange","--")]
plt.figure(figsize=(13,5))
for actor,country,color,ls in TARGETS:
    sub = g[(g["actor"]==actor) & (g["country"]==country)].sort_values("month")
    if sub.empty: continue
    plt.plot(sub["month"], sub["n_l3_sents"].astype(float),
             label=f"{actor}-{country}", color=color, linestyle=ls, linewidth=1.4)
plt.xticks(rotation=60, ha="right", fontsize=8)
plt.title("Layer3 Disruption Intensity (disruption sentence count)")
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_layer3_intensity_timeseries_labeled.png", dpi=150); plt.close()
print("Saved fig_layer3_intensity_timeseries_labeled.png")


In [ ]:
# Co-occurrence heatmaps (Layer 3 × value term, by country)
def explode_col(df, col):
    return df[col].fillna("").astype(str).apply(lambda x: [t.strip() for t in x.split(",") if t.strip()])

l3["l3_list"]  = explode_col(l3, "l3_terms")
l3["val_list"] = explode_col(l3, "value_terms")

co_rows = []
for _, r in l3.iterrows():
    for a in r["l3_list"]:
        for b in r["val_list"]:
            co_rows.append((r["country"], a, b))

if co_rows:
    co = pd.DataFrame(co_rows, columns=["country","l3_term","value_term"])
    for country in co["country"].unique():
        sub = co[co["country"] == country]
        mat = sub.pivot_table(index="l3_term", columns="value_term",
                              values="country", aggfunc="count", fill_value=0)
        fig, ax = plt.subplots(figsize=(max(6, len(mat.columns)), max(4, len(mat))))
        im = ax.imshow(mat.values.astype(float), aspect="auto", cmap="Blues")
        ax.set_xticks(range(len(mat.columns))); ax.set_xticklabels(mat.columns, rotation=45, ha="right", fontsize=9)
        ax.set_yticks(range(len(mat.index))); ax.set_yticklabels(mat.index, fontsize=9)
        plt.colorbar(im, ax=ax)
        ax.set_title(f"Layer3 × Value Co-occurrence: {country}")
        plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/fig_layer3_value_cooccur_{country}.png", dpi=150); plt.close()
        print(f"Saved fig_layer3_value_cooccur_{country}.png")
else:
    print("No co-occurrences found.")
